# 02. Data Acquisition & Management
##### DSR Berlin | Data Science Fundamentals
##### Inspired by Adam Green (adgefficiency.com), Rachel Berryman, and Samson Afolabi

---

## 🎯 Learning Objectives
By the end of this notebook, you will be able to:
1. Identify what constitutes the **"right data"** (features, targets, labels, ground-truth delays).
2. Distinguish and handle all core **tabular data formats** (continuous, categorical, ordinal, binary, time-series).
3. Evaluate critical data collection questions (volume, class balance, signal-to-noise ratio, external data).
4. Acquire data programmatically across four key modalities:
   - **Flat files** (CSV, Parquet, JSON) with performance benchmarking
   - **REST APIs** using `requests` with robust error handling
   - **Web scraping** using `BeautifulSoup` ethically
   - **Relational SQL databases** using Python's built-in `sqlite3`
5. Efficiently manage **larger-than-RAM datasets** using chunking and dtype optimization.
6. Design and evaluate **data pipelines** across the 4 stages: Ingestion, Transformation, Storage, and Consumption.


## 1. What is the "Right Data"?

Machine learning models are mathematical functions that map input features $X$ to target labels $y$:

$$f(X) \approx y$$

If your features $X$ contain no predictive signal regarding $y$, no algorithm—regardless of complexity—can learn meaningful patterns.

<div align="center">
  <img src="./images/collecting_data_questions.jpg" width="650" alt="Collecting Data Questions" />
  <p><em>Figure 1: Core considerations when sourcing data.</em></p>
</div>

### Target ($y$) and Features ($X$):
- **Features ($X$)**: The explanatory variables available *at the time prediction is needed*. (e.g., tenure, monthly charges, customer contract type).
- **Target ($y$)**: The ground-truth outcome to predict. (e.g., did the customer churn? $1 = \text{Yes}, 0 = \text{No}$).

### Critical Pitfalls in Defining Ground Truth:
1. **Label Latency / Delay**: If churn is defined as "cancelled within 90 days", you cannot use accounts opened 10 days ago—their ground truth is unknown!
2. **Selection Bias & Survivor Bias**: Only analyzing currently active customers hides why past users left.
3. **Target Leakage**: Features that inadvertently incorporate future knowledge that would not exist at inference time.


## 2. Formats of Data: The Tabular Typology

Tabular data arrives in five distinct mathematical formats. Treating one format as another (e.g. treating an ordinal rating as a nominal category, or treating a category as a continuous number) is a frequent source of modeling bugs.

<div align="center">
  <img src="./images/tabular_data_formats.jpg" width="600" alt="Tabular Data Formats" />
  <p><em>Figure 2: Formats of Tabular Data.</em></p>
</div>

### Detailed Breakdown & Real-World Examples:

| Format | Mathematical Nature | Definition | Real-World Examples | Appropriate ML Treatment |
| :--- | :--- | :--- | :--- | :--- |
| **Continuous** | Ratio / Interval | Numeric values on an unbroken continuum with infinite possible divisions. | • Monthly charges (€79.50)<br>• Account tenure (24.5 months)<br>• Temperature (21.4°C) | Scaling (`StandardScaler`), imputation by median/mean, log transforms. |
| **Categorical (Nominal)** | Qualitative / Discrete | Discrete categories with **no intrinsic order or ranking**. | • Payment method (Credit card, Bank transfer, PayPal)<br>• Device OS (iOS, Android)<br>• City (Berlin, Paris) | One-hot encoding, binary encoding, target encoding. |
| **Ordinal** | Qualitative / Ordered | Discrete categories with a **meaningful, defined ranking**, but unequal distances between ranks. | • Customer satisfaction tier (Bronze < Silver < Gold)<br>• Education level (High School < Bachelor < PhD)<br>• Survey scale (Disagree < Neutral < Agree) | Integer rank encoding (`OrdinalEncoder`), preserved monotonicity. |
| **Binary** | Dichotomous | Boolean variables having exactly two mutually exclusive states. | • Churn status (True / False)<br>• Senior citizen (0 / 1)<br>• Paperless billing (Yes / No) | Encoded as binary integers $\{0, 1\}$. |
| **Time-Series / Temporal** | Sequential / Timestamps | Records indexed by chronological timestamps with seasonality and trends. | • Customer sign-up timestamp<br>• Daily call minutes<br>• Hourly ride requests | Feature extraction (day of week, hour, cyclical sine/cosine), lag features. |


## 3. Five Critical Questions Before Collecting Data

<div align="center">
  <img src="./images/data_considerations.jpg" width="650" alt="Data Considerations" />
  <p><em>Figure 3: Key questions to evaluate when planning data acquisition.</em></p>
</div>

### 1. Do I need data from external sources?
- Does internal database data tell the whole story?
- *Example*: In predicting bike rental demand or retail foot traffic, internal sales logs alone are insufficient—you need external **weather API feeds**, **public holiday calendars**, and **local transit disruptions**.

### 2. Do I actually need to use all available features and tables?
- Sourcing 500 columns because they exist introduces the **Curse of Dimensionality**, increases data storage costs, and injects uninformative noise. Start lean with a high-conviction feature hypothesis.

### 3. Is this data likely to help the model learn?
- Historical data must reflect future operating conditions.
- *Beware of Concept Drift*: Data collected during Covid lockdown may not predict post-lockdown mobility patterns.

### 4. Do you have enough volume ($N$)?
- As a rule of thumb for tabular models: you need at least $10 \times$ to $50 \times$ more samples than features ($N \gg 50 \times P$). For complex non-linear ensembles, thousands of records are required.

### 5. Do you have enough positive labels?
- In rare-event modeling (fraud detection at 0.05%, cyber intrusions at 0.001%), a dataset of 1,000 samples might only contain 1 positive case! You need sufficient absolute counts of the minority class to validate models with confidence.


## 4. Curated Public & Institutional Data Repositories

When building portfolio projects, benchmarking prototypes, or practicing data science, leverage these premier open repositories:

### 🏛️ Research & Institutional Archives:
- [**OpenML**](https://www.openml.org/) — Thousands of standardized machine learning datasets with automated benchmark leaderboards.
- [**UCI Machine Learning Repository**](https://archive.ics.uci.edu/) — The historical gold standard for benchmark datasets (Adult Census, Iris, Wine, Heart Disease).
- [**StatLib Datasets Archive**](http://lib.stat.cmu.edu/datasets/) — Carnegie Mellon University's statistical archive.

### 🏙️ Open Government & Civic Portals:
- [**Berlin Open Data Portal (Datenportal Berlin)**](https://daten.berlin.de/) — Real-world local datasets: bicycle counts, air quality, transit delays, demographic distributions.
- [**GovData.de**](https://www.govdata.de/) — The federal open data portal for Germany.
- [**Data.gov**](https://data.gov/) — US Federal government open portal (over 300,000 datasets).

### 🏆 Competitions & Community Platforms:
- [**Kaggle Datasets**](https://www.kaggle.com/datasets) — The largest collaborative community data hub.
- [**Hugging Face Datasets**](https://huggingface.co/datasets) — The premier repository for NLP, audio, multimodal, and modern tabular benchmarks.
- [**Google Dataset Search**](https://datasetsearch.research.google.com/) — Search engine across thousands of academic and commercial data repositories.


## 5. Modality 1: Reading Flat Files (CSV vs Parquet vs JSON)

In production, CSVs are ubiquitous for human readability, but **Apache Parquet** is the enterprise standard for data lakes and analytical queries (columnar storage, built-in compression, and embedded schema).


In [1]:
import pandas as pd
import numpy as np
import time

# Let's inspect our primary project dataset
raw_csv_path = "data/telco_customer_churn_raw.csv"

# Load via Pandas
start = time.time()
df = pd.read_csv(raw_csv_path)
csv_time = time.time() - start

print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns in {csv_time:.4f} seconds.")
print(df.head(3))

Loaded 7068 rows and 21 columns in 0.0259 seconds.
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  8272-ONJLV    Male              0      No         No      12          Yes   
1  7129-CAKJW  Female              0      No         No      17          Yes   
2  7190-XHTWJ  Female              0      No         No      38          Yes   

  MultipleLines InternetService       OnlineSecurity  ...  \
0            No     Fiber optic                   No  ...   
1           Yes     Fiber optic                   No  ...   
2            No              No  No internet service  ...   

      DeviceProtection          TechSupport          StreamingTV  \
0                  Yes                   No                  Yes   
1                  Yes                   No                   No   
2  No internet service  No internet service  No internet service   

       StreamingMovies        Contract PaperlessBilling  \
0                  Yes        One year             

In [2]:
# Compare CSV vs Parquet storage efficiency
parquet_path = "data/telco_benchmark.parquet"
df.to_parquet(parquet_path, index=False)

import os

csv_size_kb = os.path.getsize(raw_csv_path) / 1024
parquet_size_kb = os.path.getsize(parquet_path) / 1024

print(f"CSV File Size:     {csv_size_kb:.2f} KB")
print(
    f"Parquet File Size: {parquet_size_kb:.2f} KB (Compression ratio: {csv_size_kb/parquet_size_kb:.1f}x)"
)

# Clean up benchmark file
os.remove(parquet_path)

CSV File Size:     957.65 KB
Parquet File Size: 198.47 KB (Compression ratio: 4.8x)


## 6. Modality 2: Ingesting Data from REST APIs

APIs (Application Programming Interfaces) allow applications to exchange live data over HTTP.
Most modern APIs return **JSON (JavaScript Object Notation)**, which maps directly to Python dictionaries and Pandas DataFrames.


In [9]:
import requests

api_url = "https://jsonplaceholder.typicode.com/users"

# Send a GET request to the API
response = requests.get(api_url)

data = response.json()
print(data)

[{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}, {'id': 2, 'name': 'Ervin Howell', 'username': 'Antonette', 'email': 'Shanna@melissa.tv', 'address': {'street': 'Victor Plains', 'suite': 'Suite 879', 'city': 'Wisokyburgh', 'zipcode': '90566-7771', 'geo': {'lat': '-43.9509', 'lng': '-34.4618'}}, 'phone': '010-692-6593 x09125', 'website': 'anastasia.net', 'company': {'name': 'Deckow-Crist', 'catchPhrase': 'Proactive didactic contingency', 'bs': 'synergize scalable supply-chains'}}, {'id': 3, 'name': 'Clementine Bauch', 'username': 'Samantha', 'email': 'Nathan@yesenia.net', 'address': {'street': 'Douglas Exten

In [ ]:
# API endpoint (a RESTful URL)
url = "https://catfact.ninja/fact"

# Send a GET request to the API
response = requests.get(url)

# Convert the response (JSON) into a Python dictionary
data = response.json()

# Print the result
print("Random cat fact:", data["fact"])

## 7. Modality 3: Web Scraping (Parsing HTML)

When data has no downloadable CSV or official API, web scraping extracts structured tables from raw HTML pages.

> ⚠️ **Ethical & Legal Scraping Rules**:
> 1. Always check the site's `robots.txt` (e.g. `example.com/robots.txt`).
> 2. Add respectful rate delays between requests (`time.sleep()`).
> 3. Do not scrape behind authenticated logins without express permission.


In [10]:
from bs4 import BeautifulSoup

html = requests.get("https://books.toscrape.com/").text
soup = BeautifulSoup(html, "html.parser")
titles = [book.h3.a["title"] for book in soup.select(".product_pod")]
titles[:5]

['A Light in the Attic',
 'Tipping the Velvet',
 'Soumission',
 'Sharp Objects',
 'Sapiens: A Brief History of Humankind']

## 8. Modality 4: Relational SQL Databases

In 90% of data science positions, data originates in a SQL database (PostgreSQL, Snowflake, MySQL, BigQuery). Python's built-in `sqlite3` lets us write and query relational databases natively.


In [11]:
import sqlite3

# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Store our DataFrame as a SQL table
df[["customerID", "gender", "Contract", "MonthlyCharges", "Churn"]].to_sql(
    "customers", conn, index=False, if_exists="replace"
)

# Execute an analytical SQL query
sql_query = """
SELECT 
    Contract,
    COUNT(*) as total_customers,
    ROUND(AVG(CAST(MonthlyCharges as FLOAT)), 2) as avg_monthly_charges
FROM customers
WHERE MonthlyCharges != '-999.0'
GROUP BY Contract
ORDER BY total_customers DESC;
"""

df_sql_result = pd.read_sql_query(sql_query, conn)
print("Analytical Query Result from SQL:")
print(df_sql_result)

conn.close()

Analytical Query Result from SQL:
             Contract  total_customers  avg_monthly_charges
0      Month-to-month             3749                66.80
1            Two year             1647                60.75
2            One year             1435                65.14
3    Month-to-month                 88                63.20
4          One year                 31                64.70
5          Two year                 22                98.83


## 9. Handling Larger-than-RAM Datasets

What happens when your dataset is 50 GB and your machine has 16 GB of RAM?
If you run `pd.read_csv("huge_file.csv")`, your operating system will crash with an **Out Of Memory (OOM)** error.

### Strategy 1: Chunked Streaming
Process data in batches using `chunksize`:


In [12]:
# Stream dataset in chunks of 1,000 rows without loading everything at once
running_total = 0.0
valid_rows = 0

for chunk in pd.read_csv(raw_csv_path, chunksize=1000):
    # Filter valid numeric values
    numeric_charges = pd.to_numeric(chunk["MonthlyCharges"], errors="coerce")
    clean_charges = numeric_charges[numeric_charges > 0]

    running_total += clean_charges.sum()
    valid_rows += len(clean_charges)

avg_charge_streamed = running_total / valid_rows
print(
    f"Computed average monthly charge via stream: €{avg_charge_streamed:.2f} across {valid_rows} rows."
)

Computed average monthly charge via stream: €65.09 across 6971 rows.


In [13]:
# Strategy 2: Memory Optimization via Column Dtypes
df_sample = pd.read_csv(raw_csv_path)
mem_before = df_sample.memory_usage(deep=True).sum() / 1024

# Downcast objects with few unique values to 'category'
categorical_cols = ["gender", "Contract", "PaymentMethod", "InternetService"]
for col in categorical_cols:
    df_sample[col] = df_sample[col].astype("category")

mem_after = df_sample.memory_usage(deep=True).sum() / 1024
print(f"Memory before optimization: {mem_before:.2f} KB")
print(
    f"Memory after categorical conversion: {mem_after:.2f} KB (Saved {((mem_before - mem_after)/mem_before)*100:.1f}%)"
)

Memory before optimization: 1915.06 KB
Memory after categorical conversion: 1438.52 KB (Saved 24.9%)


## 10. Data Pipelines: The Four Stages

A **data pipeline** is an automated software system that extracts data from its source, applies transformations, stores it in an analytical format, and delivers it to downstream consumers.

```
┌─────────────────┐     ┌───────────────────────┐     ┌────────────────────┐     ┌──────────────────────┐
│  1. Ingestion   │ ──► │   2. Transformation   │ ──► │     3. Storage     │ ──► │ 4. Analysis / Serving│
└─────────────────┘     └───────────────────────┘     └────────────────────┘     └──────────────────────┘
 (Kafka / Fivetran)       (dbt / PySpark / Polars)      (Snowflake / Parquet)       (Model API / Tableau)
```

### The 4 Stages with Real-World Industry Examples:

1. **Ingestion**:
   - Extracting raw events from operational databases, event streams, or third-party webhooks.
   - *Tools*: Apache Kafka, AWS Kinesis, Airbyte, Fivetran.
   - *Example*: Ingesting clickstream logs from mobile app sessions into raw S3 buckets.

2. **Transformation**:
   - Validating schemas, parsing timestamps, deduplicating, aggregating, and joining disparate sources.
   - *Tools*: dbt (data build tool), Apache Spark, SQL, Pandas.
   - *Example*: Cleaning raw payment logs and joining them with customer account tables.

3. **Storage**:
   - Persisting cleaned, partitioned data optimized for fast analytical scanning.
   - *Tools*: Snowflake, Google BigQuery, PostgreSQL, AWS Redshift, Delta Lake.
   - *Example*: Storing partitioned Parquet files organized by `year/month/day`.

4. **Analysis & Consumption**:
   - Delivering data to end consumers: business dashboards, automated alerts, or ML training pipelines.
   - *Tools*: Tableau, Power BI, FastAPI, MLflow, Scikit-learn.
   - *Example*: Exposing an API endpoint that predicts churn probabilities in real time for customer service agents.

### Why Data Pipelines Are Essential:
- **Data Lineage**: Tracking the complete historical pedigree of where a data point originated.
- **Data Freshness & Automation**: Automating nightly or real-time model retraining.
- **Schema Enforcement & Observability**: Alerting data engineers immediately when an upstream API alters a column name before bad data contaminates machine learning models.

---

In the next notebook, **03. Data Cleaning**, we dive deep into cleaning our raw Telco Churn dataset: isolating test sets, tackling disguised missing values, analyzing duplicates, and treating outliers!
